In [1]:
import time
import json
import math
import numpy as np
from itertools import combinations
import chipsplitting as cs
from chipsplitting.hyperfield import (
    HyperfieldHomogeneousLinearSystem as HVLinearSystem,
    HyperfieldVector
)
from chipsplitting import pairing_matrix, print_matrix, PascalForm
import chipsplitting.hyperfield.utils as utils


In [8]:
# utils
def save_fundamental_models(model, json_file_name):
    with open(json_file_name, 'w') as json_file:
        json.dump([{"support": support, "solution": [x for x in solution] } for support, solution in model], json_file, default=float)
    
    print(f"JSON dump has been written to {json_file_name}")

def load_fundamental_models(json_file_name):
    with open(json_file_name, 'r') as json_file:
        data = json.load(json_file)
        return [(item["support"], item["solution"]) for item in data]

def expand_expression(expression):
    if expression.startswith('t^'):
        power = int(expression[2:])
        return [1 if k == power else 0 for k in range(power + 1)]
    elif expression.startswith('(1-t)^'):
        power = int(expression[len('(1-t)^'):])
        return [math.comb(power,k) * (-1)**(k) for k in range(power + 1)]
    else:
        raise ValueError("Invalid expression")

def multiply_expression(left_exp: list[int], right_exp: list[int]):
    left_power = len(left_exp) - 1
    right_power = len(right_exp) - 1
    degree = left_power + right_power
    t_coefficients = [0] * (degree + 1)
    for a in range(left_power + 1):
        for b in range(right_power + 1):
            t_coefficients[a + b] += left_exp[a] * right_exp[b]
    return t_coefficients

def pad_with_zero(list, desired_length: int):
    if len(list) < desired_length:
        return list + [0] * (desired_length - len(list))
    return list

def create_matrix_from_support(degree: int, support: list[int]):
    support = [utils.to_coordinate(index) for index in support]
    A = np.array([pad_with_zero(multiply_expression(expand_expression(f"t^{col}"), expand_expression(f"(1-t)^{row}")), degree + 1) for col, row in support])
    return A.T

def is_fundamental(support_as_index, degree):
    A = create_matrix_from_support(degree, support_as_index)
    b = np.array([1] + [0] * degree)
    sol, norm, rank, singular_values = np.linalg.lstsq(A, b, rcond=None)

    if rank < len(support_as_index):
        return None

    if len(norm) == 0 or np.isclose(norm[0], 0):
        return sol
    
    return None

def apply_symmetry(config):
    def swap(tuple):
        return (tuple[1], tuple[0])
    def to_array_index(tuple):
        return utils.get_array_index(tuple[0], tuple[1])
    return tuple(sorted([to_array_index(swap(utils.to_coordinate(index))) for index in config]))

### Compute possible supports of fundamental models

In [3]:
def find_positive_supports(pos_support_size, degree_end = None, expand = True):
    degree_end = degree_end if degree_end is not None else 2 * pos_support_size - 3
    # hyperfield variety generated by all pascal equations
    S = {}
    for d in range(1, degree_end + 1):
        start = time.time()
        num_cells = cs.utils.gauss(d+1)
        # positive cells are all cells except the (0,0) cell
        num_pos_cells = num_cells - 1
        # skip if the degree has not enough cells
        if num_pos_cells < pos_support_size:
            continue
            
        base_types = ["diag", "row", "col"]
        A = [PascalForm(d, b, k).to_hyperfield() for b in base_types for k in range(d + 1)]
        linear_system = HVLinearSystem(A)
        

        solutions = linear_system.quick_solve_loop(pos_support_size)
        
        if expand:
            # optimizable
            def expand_solution(sol):
                num_selections = pos_support_size - len(sol)
                domain = [x for x in range(cs.utils.gauss(d + 1)) if x not in sol]
                return [tuple(sorted(sol + c)) for c in combinations(domain, num_selections)]

            expanded = []
            for sol in solutions:
                if len(sol) < pos_support_size:
                    expanded += expand_solution(sol)
                else:
                    expanded.append(sol)
            solutions = list(set(tuple(expanded)))
        
        S[d] = solutions
        print(f"d={d}, #supports={len(solutions)}. Elapsed: {time.time() - start}")
    return S


In [18]:
possible_supports = {}
n_begin = 1
n_end = 6
for pos_support_size in range(n_begin + 1, n_end + 2):
    print(f"n={pos_support_size - 1}")
    possible_supports[pos_support_size] = find_positive_supports(pos_support_size)
    #possible_supports[pos_support_size] = find_positive_supports(pos_support_size, mode = "fast", expand = True)
print("Reducing possible solutions by applying symmetry...")

possible_supports_copy = {}

for n, d_to_solutions in possible_supports.items():
    d_to_solutions_copy = {}
    for d, solutions in d_to_solutions.items():
        solutions_copy = set()
        for support in solutions:
            if tuple(sorted(cs.reflect_support(support))) not in solutions_copy:
                solutions_copy.add(support)
        d_to_solutions_copy[d] = list(solutions_copy)
    possible_supports_copy[n] = d_to_solutions_copy

possible_supports = possible_supports_copy

for n, d_to_solutions in possible_supports.items():
    print("")
    print(f"n={n-1}")
    for d, solutions in d_to_solutions.items():
        print(f"d={d}, #supports={len(solutions)}")

n=1
d=1, #supports=1. Elapsed: 0.0002551078796386719
n=2
d=2, #supports=3. Elapsed: 0.0002942085266113281
d=3, #supports=1. Elapsed: 0.00037598609924316406
n=3
d=2, #supports=7. Elapsed: 0.00025010108947753906
d=3, #supports=20. Elapsed: 0.00037980079650878906
d=4, #supports=16. Elapsed: 0.0005691051483154297
d=5, #supports=6. Elapsed: 0.0008037090301513672
n=4
d=2, #supports=5. Elapsed: 0.0002779960632324219
d=3, #supports=70. Elapsed: 0.0004589557647705078
d=4, #supports=233. Elapsed: 0.0008718967437744141
d=5, #supports=437. Elapsed: 0.0017058849334716797
d=6, #supports=645. Elapsed: 0.003136873245239258
d=7, #supports=784. Elapsed: 0.005414009094238281
n=5
d=3, #supports=108. Elapsed: 0.0006897449493408203
d=4, #supports=1038. Elapsed: 0.002676248550415039
d=5, #supports=4402. Elapsed: 0.00864100456237793
d=6, #supports=12318. Elapsed: 0.022944927215576172
d=7, #supports=26325. Elapsed: 0.047650814056396484
d=8, #supports=46655. Elapsed: 0.11530184745788574
d=9, #supports=77426. El

In [5]:
def is_positive(solution):
    for x in solution:
        if np.isclose(x,0):
            return False
        if x < 0:
            return False
    return True

def is_parametric(solution):
    for x in solution:
        if not x.is_number:
            return True
    return False

def is_asymmetric(support):
    return list(sorted(support)) != list(sorted(apply_symmetry(support)))

def filter_fundamental_models(fundamental_models):
    return [(support, solution) for support, solution in fundamental_models if solution and is_positive(solution)]

def find_fundamental_models(n, degrees):
    if isinstance(degrees, int):
        degrees = [degrees]
        
    fundamental_models = []
    
    for d in degrees:
        start = time.time()
        supports_to_check = possible_supports[n + 1][d]

        print("")
        print(f"d={d}: {len(supports_to_check)} supports to check")

        for index, support in enumerate(supports_to_check):
            if index == 100:
                elapsed = time.time() - start
                print(f"Solved linear equations of 100 supports in {round(elapsed, 2)}s. Estimate: {round(len(supports_to_check) / 100 * elapsed / 60, 2)}min")
                
            solution = is_fundamental(support, d)
            fundamental_models.append((support, solution))

        end = time.time()
        print(f"Solved linear equations for all supports. Elapsed time: {round(end - start, 2)}s")

    print()
    
    fundamental_models = [(support, solution) for support, solution in fundamental_models if solution is not None]
    print(f"After excluding supports with no solutions, {len(fundamental_models)} fundamental models are left.")

    #fundamental_models = [(support, solution) for support, solution in fundamental_models if not is_parametric(solution)]
    #print(f"After excluding supports with parametric solutions, {len(fundamental_models)} fundamental models are left.")

    fundamental_models = [(support, solution) for support, solution in fundamental_models if is_positive(solution)]
    print(f"After excluding supports with non-positive solutions, {len(fundamental_models)} fundamental models are left.")

    print(f"")
    print(f"We have found {len(fundamental_models)} fundamental models.")

    missing_models_from_symmetry = [(apply_symmetry(support), solution) for support, solution in fundamental_models if is_asymmetric(support)]
    print(f"Found additional {len(missing_models_from_symmetry)} fundamental models due to symmetry.")
    
    return fundamental_models + missing_models_from_symmetry

## $n = 3$

In [37]:
fundamental_models = find_fundamental_models(2, [2])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=2: 2 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 2 fundamental models are left.
After excluding supports with non-positive solutions, 2 fundamental models are left.

We have found 2 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 3 fundamental models ✨


In [36]:
fundamental_models = find_fundamental_models(2, [3])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=3: 1 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 1 fundamental models are left.
After excluding supports with non-positive solutions, 1 fundamental models are left.

We have found 1 fundamental models.
Found additional 0 fundamental models due to symmetry.

There exist exactly 1 fundamental models ✨


## $n = 4$

In [30]:
fundamental_models = find_fundamental_models(3, [3])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=3: 12 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 11 fundamental models are left.
After excluding supports with non-positive solutions, 7 fundamental models are left.

We have found 7 fundamental models.
Found additional 5 fundamental models due to symmetry.

There exist exactly 12 fundamental models ✨


In [33]:
fundamental_models = find_fundamental_models(3, [4])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=4: 8 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 2 fundamental models are left.
After excluding supports with non-positive solutions, 2 fundamental models are left.

We have found 2 fundamental models.
Found additional 2 fundamental models due to symmetry.

There exist exactly 4 fundamental models ✨


In [32]:
fundamental_models = find_fundamental_models(3, [5])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=5: 4 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 1 fundamental models are left.
After excluding supports with non-positive solutions, 1 fundamental models are left.

We have found 1 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 2 fundamental models ✨


## $n = 5$

In [26]:
fundamental_models = find_fundamental_models(4, [4])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=4: 119 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.01s

After excluding supports with no solutions, 108 fundamental models are left.
After excluding supports with non-positive solutions, 43 fundamental models are left.

We have found 43 fundamental models.
Found additional 39 fundamental models due to symmetry.

There exist exactly 82 fundamental models ✨


In [27]:
fundamental_models = find_fundamental_models(4, [5])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=5: 227 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.02s

After excluding supports with no solutions, 55 fundamental models are left.
After excluding supports with non-positive solutions, 23 fundamental models are left.

We have found 23 fundamental models.
Found additional 15 fundamental models due to symmetry.

There exist exactly 38 fundamental models ✨


In [28]:
fundamental_models = find_fundamental_models(4, [6])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=6: 326 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.03s

After excluding supports with no solutions, 10 fundamental models are left.
After excluding supports with non-positive solutions, 5 fundamental models are left.

We have found 5 fundamental models.
Found additional 5 fundamental models due to symmetry.

There exist exactly 10 fundamental models ✨


In [29]:
fundamental_models = find_fundamental_models(4, [7])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=7: 402 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.04s

After excluding supports with no solutions, 7 fundamental models are left.
After excluding supports with non-positive solutions, 3 fundamental models are left.

We have found 3 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 4 fundamental models ✨


## $n = 6$

In [22]:
fundamental_models = find_fundamental_models(5, [5])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=5: 2232 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.1s

After excluding supports with no solutions, 1861 fundamental models are left.
After excluding supports with non-positive solutions, 306 fundamental models are left.

We have found 306 fundamental models.
Found additional 296 fundamental models due to symmetry.

There exist exactly 602 fundamental models ✨


In [23]:
fundamental_models = find_fundamental_models(5, [6])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=6: 6177 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.02min
Solved linear equations for all supports. Elapsed time: 0.25s

After excluding supports with no solutions, 860 fundamental models are left.
After excluding supports with non-positive solutions, 129 fundamental models are left.

We have found 129 fundamental models.
Found additional 125 fundamental models due to symmetry.

There exist exactly 254 fundamental models ✨


In [23]:
fundamental_models = find_fundamental_models(5, [7])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=6: 6177 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.02min
Solved linear equations for all supports. Elapsed time: 0.25s

After excluding supports with no solutions, 860 fundamental models are left.
After excluding supports with non-positive solutions, 129 fundamental models are left.

We have found 129 fundamental models.
Found additional 125 fundamental models due to symmetry.

There exist exactly 254 fundamental models ✨


In [23]:
fundamental_models = find_fundamental_models(5, [7])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=6: 6177 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.02min
Solved linear equations for all supports. Elapsed time: 0.25s

After excluding supports with no solutions, 860 fundamental models are left.
After excluding supports with non-positive solutions, 129 fundamental models are left.

We have found 129 fundamental models.
Found additional 125 fundamental models due to symmetry.

There exist exactly 254 fundamental models ✨


In [24]:
fundamental_models = find_fundamental_models(5, [8])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=8: 23353 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.05min
Solved linear equations for all supports. Elapsed time: 0.92s

After excluding supports with no solutions, 512 fundamental models are left.
After excluding supports with non-positive solutions, 12 fundamental models are left.

We have found 12 fundamental models.
Found additional 12 fundamental models due to symmetry.

There exist exactly 24 fundamental models ✨


In [25]:
fundamental_models = find_fundamental_models(5, [9])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")


d=9: 38838 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 0.11min
Solved linear equations for all supports. Elapsed time: 1.52s

After excluding supports with no solutions, 537 fundamental models are left.
After excluding supports with non-positive solutions, 1 fundamental models are left.

We have found 1 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 2 fundamental models ✨


## $n = 7$

### $d = 3,4,5$

In [6]:
fundamental_models = find_fundamental_models(6, [3,4,5])
print()
print(f"There exist exactly {len(fundamental_models)} fundamental models ✨")



d=3: 48 supports to check
Solved linear equations for all supports. Elapsed time: 0.01s

d=4: 1226 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.05s

d=5: 10635 supports to check
Solved linear equations of 100 supports in 0.0s. Estimate: 0.01min
Solved linear equations for all supports. Elapsed time: 0.64s

After excluding supports with no solutions, 0 fundamental models are left.
After excluding supports with non-positive solutions, 0 fundamental models are left.

We have found 0 fundamental models.
Found additional 0 fundamental models due to symmetry.

There exist exactly 0 fundamental models ✨


### $d = 6$

In [9]:
n = 6
d = 6
fundamental_models = find_fundamental_models(n, d)
print(f"Found {len(fundamental_models)} potential fundamental models")


d=6: 52284 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.13min
Solved linear equations for all supports. Elapsed time: 1.77s

After excluding supports with no solutions, 41359 fundamental models are left.
After excluding supports with non-positive solutions, 3370 fundamental models are left.

We have found 3370 fundamental models.
Found additional 3340 fundamental models due to symmetry.
Found 6710 potential fundamental models


### $d = 7$

In [10]:
n = 6
d = 7
fundamental_models = find_fundamental_models(n, d)
print(f"Found {len(fundamental_models)} potential fundamental models")



d=7: 180527 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 0.47min
Solved linear equations for all supports. Elapsed time: 6.94s

After excluding supports with no solutions, 27446 fundamental models are left.
After excluding supports with non-positive solutions, 1250 fundamental models are left.

We have found 1250 fundamental models.
Found additional 1171 fundamental models due to symmetry.
Found 2421 potential fundamental models


### $d = 8$

In [11]:
n = 6
d = 8
fundamental_models = find_fundamental_models(n, d)
print(f"Found {len(fundamental_models)} potential fundamental models")




d=8: 491601 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 1.34min
Solved linear equations for all supports. Elapsed time: 19.17s

After excluding supports with no solutions, 31508 fundamental models are left.
After excluding supports with non-positive solutions, 324 fundamental models are left.

We have found 324 fundamental models.
Found additional 319 fundamental models due to symmetry.
Found 643 potential fundamental models


### $d = 9$

In [13]:
n = 6
d = 9
fundamental_models = find_fundamental_models(n, d)
print(f"Found {len(fundamental_models)} potential fundamental models")




d=9: 1160287 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 2.06min
Solved linear equations for all supports. Elapsed time: 49.53s

After excluding supports with no solutions, 47093 fundamental models are left.
After excluding supports with non-positive solutions, 106 fundamental models are left.

We have found 106 fundamental models.
Found additional 92 fundamental models due to symmetry.
Found 198 potential fundamental models


### $d = 10$

In [15]:
n = 6
d = 10
fundamental_models = find_fundamental_models(n, d)
print(f"Found {len(fundamental_models)} potential fundamental models")




d=10: 2427353 supports to check
Solved linear equations of 100 supports in 0.0s. Estimate: 1.94min
Solved linear equations for all supports. Elapsed time: 107.19s

After excluding supports with no solutions, 68417 fundamental models are left.
After excluding supports with non-positive solutions, 16 fundamental models are left.

We have found 16 fundamental models.
Found additional 16 fundamental models due to symmetry.
Found 32 potential fundamental models


### $d = 11$

In [14]:
n = 6
d = 11
fundamental_models = find_fundamental_models(n, d)
print(f"Found {len(fundamental_models)} potential fundamental models")




d=11: 4710754 supports to check
Solved linear equations of 100 supports in 0.0s. Estimate: 3.58min
Solved linear equations for all supports. Elapsed time: 212.26s

After excluding supports with no solutions, 99262 fundamental models are left.
After excluding supports with non-positive solutions, 2 fundamental models are left.

We have found 2 fundamental models.
Found additional 2 fundamental models due to symmetry.
Found 4 potential fundamental models
